# Análisis de Datos — Módulo 1: Tomando confianza con Databricks

En este notebook practicaremos los fundamentos de **Apache Spark sobre Databricks**:

1. Crear un **Volume** dentro de **Unity Catalog** para almacenar ficheros.
2. Descargar varios CSV y cargarlos en el Volume.
3. Leer esos CSV en un **DataFrame** de Spark.
4. Definir un **esquema explícito** (`StructType`) para tipar las columnas correctamente.
5. Consultar los datos con **SQL** mediante una vista temporal.
6. Visualizar resultados con **Matplotlib** y **Seaborn**.

> **Unity Catalog** es la capa de gobernanza de datos de Databricks. Los **Volumes** son objetos de Unity Catalog que sirven para almacenar ficheros no tabulares (CSV, JSON, imágenes, modelos, etc.) bajo el mismo modelo de permisos que las tablas. Su ruta sigue el patrón `/Volumes/<catalog>/<schema>/<volume>/...`.

## 1. Cargar el conjunto de datos CSV en un Volume de Unity Catalog

Un **Volume** es un objeto lógico de Unity Catalog que representa un volumen de almacenamiento. Existen dos tipos:

- **Managed volume**: Databricks gestiona el almacenamiento físico (recomendado para empezar).
- **External volume**: apunta a una ubicación de almacenamiento externa que tú gestionas.

La sintaxis general para crear un Volume gestionado es:

```sql
CREATE VOLUME [IF NOT EXISTS] <catalog>.<schema>.<volume_name>
```

Si no indicas catálogo y esquema, Databricks usa los activos en la sesión (`current_catalog()` y normalmente `default`).

In [ ]:
%sql
-- %sql es un "magic command" de Databricks: indica que la celda se ejecuta como SQL,
-- aunque el notebook por defecto sea Python.
--
-- CREATE VOLUME crea un Volume gestionado dentro del esquema actual (por defecto, 'default').
-- IF NOT EXISTS evita un error si el Volume ya existe.
CREATE VOLUME IF NOT EXISTS spark_lab

In [ ]:
%sql
-- Práctica 1: vuelve a escribir aquí el comando para crear el Volume.


In [ ]:
%sql
-- Práctica 2: vuelve a escribir aquí el comando para crear el Volume.


In [ ]:
import requests

# Obtenemos el catálogo activo de la sesión actual de Spark.
# current_catalog() es una función SQL de Unity Catalog que devuelve el catálogo seleccionado.
# .collect()[0][0] toma la primera fila y la primera columna del resultado.
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Construimos la ruta base del Volume usando el catálogo actual.
# Formato estándar de un Volume en Unity Catalog: /Volumes/<catalog>/<schema>/<volume>
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# Lista de archivos CSV con datos de ventas que vamos a descargar.
files = ["2019.csv", "2020.csv", "2021.csv"]

# Descargamos cada archivo desde un repositorio público de GitHub
# y lo escribimos directamente en el Volume.
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()  # Lanza excepción si la descarga falla (códigos 4xx/5xx).

    # 'wb' = escritura en modo binario (preserva el contenido byte a byte).
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

In [ ]:
# Práctica 1: reescribe aquí el código para descargar los CSV al Volume.


In [ ]:
# Práctica 2: reescribe aquí el código para descargar los CSV al Volume.


## 2. Cargar los archivos CSV en un DataFrame

Un **DataFrame** de Spark es una colección distribuida de datos organizada en columnas con nombre. Conceptualmente es como una tabla relacional, pero está diseñada para procesarse en paralelo en el clúster.

Para leer ficheros usamos el `DataFrameReader` accesible vía `spark.read`. Las dos formas equivalentes más habituales son:

```python
# Forma genérica
spark.read.load(path, format='csv')

# Forma específica
spark.read.csv(path)
```

Opciones útiles al leer CSV: `header=True` (la primera fila contiene los nombres de columna), `inferSchema=True` (Spark deduce los tipos), `sep=','` (separador).

In [ ]:
# spark.read.load lee uno o varios archivos del Volume.
# El comodín *.csv carga todos los CSV del directorio en un único DataFrame.
# format='csv' indica el formato de origen.
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*.csv', format='csv')

# display() es una función específica de Databricks que muestra el DataFrame
# en una tabla interactiva (con filtros, ordenación y gráficos integrados).
# .limit(100) restringe la salida a las primeras 100 filas.
display(df.limit(100))

In [ ]:
# Práctica 1: reescribe aquí el código para leer los CSV en un DataFrame y mostrar 100 filas.


In [ ]:
# Práctica 2: reescribe aquí el código para leer los CSV en un DataFrame y mostrar 100 filas.


## 3. Definir un esquema explícito para el DataFrame

Cuando Spark infiere el esquema automáticamente puede equivocarse en los tipos (por ejemplo, leer fechas como cadenas de texto). Además, la inferencia tiene un coste: Spark debe leer los datos dos veces (una para deducir tipos, otra para cargarlos).

Definir un **esquema explícito** con `StructType` y `StructField`:

- garantiza los tipos correctos (`IntegerType`, `DateType`, `FloatType`, etc.),
- mejora el rendimiento (Spark no infiere),
- documenta claramente la forma de los datos.

Tipos más usados de `pyspark.sql.types`: `StringType`, `IntegerType`, `LongType`, `FloatType`, `DoubleType`, `BooleanType`, `DateType`, `TimestampType`.

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# StructType describe la estructura de cada fila como una lista de StructField.
# Cada StructField indica: nombre de la columna, tipo de dato y (opcional) si admite nulos.
orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

# Volvemos a leer los CSV pero ahora aplicando el esquema definido.
# 'schema=orderSchema' le indica a Spark cómo interpretar cada columna.
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*.csv', format='csv', schema=orderSchema)
display(df.limit(100))

In [ ]:
# Práctica 1: reescribe aquí el StructType y la lectura del CSV con esquema explícito.


In [ ]:
# Práctica 2: reescribe aquí el StructType y la lectura del CSV con esquema explícito.


## 4. Consultar los datos con SQL

Spark permite ejecutar consultas SQL sobre cualquier DataFrame siempre que lo registremos como **vista temporal** (*temp view*). La vista existe sólo durante la sesión activa y desaparece al cerrarla.

Tipos de vistas:

- `createOrReplaceTempView("nombre")` — vista local de la sesión.
- `createOrReplaceGlobalTempView("nombre")` — vista accesible desde otras sesiones del mismo clúster (se consulta como `global_temp.nombre`).

Para ejecutar SQL desde Python usamos `spark.sql("...")`, que devuelve otro DataFrame.

In [ ]:
# Registramos el DataFrame como vista temporal con el nombre 'salesorders'.
# createOrReplaceTempView crea la vista o la reemplaza si ya existe con ese nombre.
df.createOrReplaceTempView("salesorders")

# spark.sql ejecuta una consulta SQL sobre las vistas/tablas registradas
# y devuelve un nuevo DataFrame con el resultado.
spark_df = spark.sql("SELECT * FROM salesorders")
display(spark_df)

In [ ]:
# Práctica 1: registra el DataFrame como vista temporal y lánzale un SELECT * con spark.sql.


In [ ]:
# Práctica 2: registra el DataFrame como vista temporal y lánzale un SELECT * con spark.sql.


In [ ]:
# Calculamos la facturación bruta (GrossRevenue) agrupada por año.
#  - YEAR(OrderDate) extrae el año de la fecha del pedido.
#  - CAST(... AS CHAR(4)) lo convierte a texto de 4 caracteres (útil para tratarlo como categoría).
#  - La facturación bruta por línea = UnitPrice * Quantity + Tax.
#  - SUM(...) suma todas las líneas de cada año.
#  - GROUP BY agrupa por año; ORDER BY ordena el resultado cronológicamente.
sqlQuery = "SELECT CAST(YEAR(OrderDate) AS CHAR(4)) AS OrderYear, \
               SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue \
        FROM salesorders \
        GROUP BY CAST(YEAR(OrderDate) AS CHAR(4)) \
        ORDER BY OrderYear"

df_spark = spark.sql(sqlQuery)

# .show() imprime el contenido del DataFrame en formato de tabla en texto plano (útil en logs).
df_spark.show()

In [ ]:
# Práctica 1: reescribe la consulta SQL que agrupa la facturación bruta por año y muéstrala.


In [ ]:
# Práctica 2: reescribe la consulta SQL que agrupa la facturación bruta por año y muéstrala.


## 5. Visualización con Matplotlib

**Matplotlib** es la librería de gráficos más usada en Python. No entiende los DataFrames de Spark (que viven distribuidos en el clúster), por lo que primero convertimos a un DataFrame de **pandas** con `.toPandas()`.

> ⚠️ `.toPandas()` trae **todos los datos al driver** del clúster. Úsalo sólo con conjuntos de datos pequeños o ya agregados (como en este caso, donde tenemos una fila por año).

In [ ]:
from matplotlib import pyplot as plt

# Matplotlib requiere un DataFrame de pandas (no de Spark).
# .toPandas() recoge los datos del clúster en memoria del driver.
df_sales = df_spark.toPandas()

# Diagrama de barras: eje X = año del pedido, eje Y = facturación bruta.
plt.bar(x=df_sales['OrderYear'], height=df_sales['GrossRevenue'])

# plt.show() renderiza el gráfico en la salida de la celda.
plt.show()

In [ ]:
# Práctica 1: convierte el DataFrame a pandas y dibuja el bar plot con Matplotlib.


In [ ]:
# Práctica 2: convierte el DataFrame a pandas y dibuja el bar plot con Matplotlib.


## 6. Visualización con Seaborn

**Seaborn** es una librería construida sobre Matplotlib que ofrece:

- una API más declarativa (`x=`, `y=`, `data=`),
- estilos visuales más cuidados por defecto,
- gráficos estadísticos avanzados (boxplot, violin, heatmap, pairplot...).

Sigue trabajando con DataFrames de pandas, así que reutilizamos `df_sales`.

In [ ]:
import seaborn as sns

# Limpiamos cualquier figura previa de Matplotlib para empezar con un lienzo en blanco.
plt.clf()

# barplot dibuja un diagrama de barras directamente desde un DataFrame de pandas,
# indicando qué columnas usar para los ejes X e Y mediante el parámetro 'data='.
ax = sns.barplot(x="OrderYear", y="GrossRevenue", data=df_sales)
plt.show()

In [ ]:
# Práctica 1: dibuja el mismo gráfico de barras usando seaborn.barplot.


In [ ]:
# Práctica 2: dibuja el mismo gráfico de barras usando seaborn.barplot.
